In [0]:
#Load all tables
from pyspark.sql import functions as F

tables = {
    "customer_master": spark.table("customer_master"),
    "product_master": spark.table("product_master"),
    "order_master": spark.table("order_master"),
    "order_line": spark.table("order_line"),
    "engagement_event": spark.table("engagement_event"),
    "support_case": spark.table("support_case"),
    "consent": spark.table("consent"),
    "segment_rule": spark.table("segment_rule"),
    "segment_membership": spark.table("segment_membership"),
    "action_catalog": spark.table("action_catalog"),
    "customer_recommendation": spark.table("customer_recommendation"),
    "action_outcome": spark.table("action_outcome")
}

print("All tables loaded successfully!")

In [0]:
#Check primary keys
primary_keys = {
    "customer_master": "customer_id",
    "product_master": "product_id",
    "order_master": "order_id",
    "order_line": "order_line_id",
    "engagement_event": "event_id",
    "support_case": "case_id",
    "consent": "consent_id",
    "segment_rule": "segment_id",
    "segment_membership": "membership_id",
    "action_catalog": "action_id",
    "customer_recommendation": "recommendation_id",
    "action_outcome": "outcome_id"
}

for table_name, pk in primary_keys.items():
    df = tables[table_name]

    total_rows = df.count()
    null_count = df.filter(F.col(pk).isNull()).count()
    duplicate_count = (
        total_rows - df.select(pk).distinct().count()
    )

    print(
        table_name,
        "| Total:", total_rows,
        "| NULL keys:", null_count,
        "| Duplicate keys:", duplicate_count
    )

In [0]:
#Check foreign keys
foreign_key_checks = [
    ("order_master", "customer_id", "customer_master", "customer_id"),
    ("order_line", "order_id", "order_master", "order_id"),
    ("order_line", "product_id", "product_master", "product_id"),
    ("engagement_event", "customer_id", "customer_master", "customer_id"),
    ("support_case", "customer_id", "customer_master", "customer_id"),
    ("consent", "customer_id", "customer_master", "customer_id"),
    ("segment_membership", "customer_id", "customer_master", "customer_id"),
    ("segment_membership", "segment_id", "segment_rule", "segment_id"),
    ("customer_recommendation", "customer_id", "customer_master", "customer_id"),
    ("customer_recommendation", "action_id", "action_catalog", "action_id"),
    ("action_outcome", "recommendation_id", "customer_recommendation", "recommendation_id")
]

for child_table, child_col, parent_table, parent_col in foreign_key_checks:

    child_df = tables[child_table]
    parent_df = tables[parent_table]

    invalid_count = (
        child_df
        .filter(F.col(child_col).isNotNull())
        .join(
            parent_df.select(parent_col).distinct(),
            child_df[child_col] == parent_df[parent_col],
            "left_anti"
        )
        .count()
    )

    print(
        child_table, "->", parent_table,
        "| Invalid foreign keys:", invalid_count
    )

In [0]:
#Check duplicate records
for table_name, df in tables.items():

    duplicate_rows = (
        df.count() - df.dropDuplicates().count()
    )

    print(table_name, "| Duplicate rows:", duplicate_rows)

In [0]:
#Check missing values
required_columns = {
    "customer_master": ["customer_id", "city", "signup_date"],
    "product_master": ["product_id", "product_name", "unit_price"],
    "order_master": ["order_id", "customer_id", "order_date", "order_status"],
    "order_line": ["order_line_id", "order_id", "product_id", "quantity"],
    "engagement_event": ["event_id", "customer_id", "event_type"],
    "support_case": ["case_id", "customer_id", "case_status"],
    "consent": ["consent_id", "customer_id"],
    "segment_membership": ["membership_id", "customer_id", "segment_id"],
    "customer_recommendation": ["recommendation_id", "customer_id"],
    "action_outcome": ["outcome_id", "recommendation_id"]
}

for table_name, columns in required_columns.items():

    df = tables[table_name]

    for column in columns:
        missing_count = df.filter(
            F.col(column).isNull()
        ).count()

        print(
            table_name,
            "|", column,
            "| Missing:", missing_count
        )

In [0]:
#Validate order and support dates
invalid_orders = tables["order_master"].filter(
    F.col("order_date") > F.lit("2026-10-30").cast("date")
)

print("Future orders:", invalid_orders.count())

In [0]:
invalid_support = tables["support_case"].filter(
    (F.col("resolved_date").isNotNull()) &
    (F.col("resolved_date") < F.col("opened_date"))
)

print("Invalid support dates:", invalid_support.count())

In [0]:
#Validate marketing consent
consent_violations = tables["customer_recommendation"].filter(
    (F.col("action_type") == "MARKETING") &
    (F.col("consent_status") != "GRANTED")
)

print("Marketing consent violations:", consent_violations.count())

In [0]:
#Validate customer memberships
invalid_memberships = (
    tables["segment_membership"]
    .filter(F.col("membership_status") == "ACTIVE")
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") != 1)
)

print("Customers with incorrect active memberships:",
      invalid_memberships.count())

In [0]:
#Create a validation summary
validation_results = []

for table_name, pk in primary_keys.items():

    df = tables[table_name]

    total_rows = df.count()
    null_keys = df.filter(F.col(pk).isNull()).count()
    duplicate_keys = total_rows - df.select(pk).distinct().count()

    validation_results.append(
        (table_name, "PRIMARY_KEY_NULL", null_keys)
    )

    validation_results.append(
        (table_name, "PRIMARY_KEY_DUPLICATE", duplicate_keys)
    )

validation_summary_df = spark.createDataFrame(
    validation_results,
    ["table_name", "check_name", "issue_count"]
)

validation_summary_df = validation_summary_df.withColumn(
    "validation_date",
    F.lit("2026-10-30").cast("date")
)

display(validation_summary_df)

In [0]:
validation_summary_df.write.format("delta").mode("overwrite").saveAsTable(
    "validation_summary"
)

print("Validation summary saved successfully!")